In [ ]:
#import necessary libraries
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
from pyspark.sql.functions import *
from pyspark.sql.types import DoubleType, IntegerType

In [2]:
!pip show pyspark
!pip show delta-spark

Name: pyspark
Version: 4.1.1
Summary: Apache Spark Python API
Home-page: https://github.com/apache/spark/tree/master/python
Author: Spark Developers
Author-email: dev@spark.apache.org
License: Apache-2.0
Location: C:\Users\Soham Deshmukh\Desktop\B.Tech\Celebal Internship\cei-data-engineering\.venv\Lib\site-packages
Requires: py4j
Required-by: delta_spark
Name: delta_spark
Version: 4.3.1
Summary: Python APIs for using Delta Lake with Apache Spark
Home-page: https://github.com/delta-io/delta/
Author: The Delta Lake Project Authors
Author-email: delta-users@googlegroups.com
License: Apache-2.0
Location: C:\Users\Soham Deshmukh\Desktop\B.Tech\Celebal Internship\cei-data-engineering\.venv\Lib\site-packages
Requires: importlib_metadata, pyspark
Required-by: 


In [ ]:
#Configure Spark Session with Delta Lake support
builder = (
    SparkSession.builder
    .master("local[*]")
    .appName("Week7_DeltaLakeMerge")
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [4]:
print(spark.version)

4.1.1


In [ ]:
#Read the original dataset into a DataFrame
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("../data/Sample - Superstore.csv")
)

df.show(10)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

### Basic EDA

In [6]:
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



In [7]:
print("Total Rows :", df.count())

Total Rows : 9994


In [8]:
print(df.columns)

['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


In [9]:
df.describe().show()

+-------+------------------+--------------+----------+---------+--------------+-----------+------------------+-----------+-------------+--------+-------+------------------+-------+---------------+----------+------------+--------------------+------------------+------------------+------------------+------------------+
|summary|            Row ID|      Order ID|Order Date|Ship Date|     Ship Mode|Customer ID|     Customer Name|    Segment|      Country|    City|  State|       Postal Code| Region|     Product ID|  Category|Sub-Category|        Product Name|             Sales|          Quantity|          Discount|            Profit|
+-------+------------------+--------------+----------+---------+--------------+-----------+------------------+-----------+-------------+--------+-------+------------------+-------+---------------+----------+------------+--------------------+------------------+------------------+------------------+------------------+
|  count|              9994|          9994|   

In [10]:
print(len(df.columns))

21


In [11]:
print(len(df.columns))

21


In [12]:
(
df.groupBy("Order ID")
.count()
.filter("count > 1")
.show()
)

+--------------+-----+
|      Order ID|count|
+--------------+-----+
|US-2017-164147|    3|
|CA-2017-132521|    3|
|CA-2015-116750|    2|
|CA-2015-115798|    4|
|US-2017-111024|    3|
|CA-2017-140326|    3|
|CA-2015-128083|    3|
|CA-2016-145730|    3|
|CA-2016-135776|    7|
|CA-2015-161830|    2|
|CA-2016-134936|    3|
|CA-2016-149783|    3|
|CA-2016-124016|    3|
|CA-2014-141838|    3|
|US-2016-125969|    2|
|CA-2014-125612|    3|
|CA-2017-149559|    3|
|CA-2016-155474|    2|
|CA-2014-168592|    3|
|CA-2017-125115|    2|
+--------------+-----+
only showing top 20 rows


In [13]:
df.select(
[count(when(col(c).isNull(),c)).alias(c)
for c in df.columns
]
).show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



In [14]:
df = df.fillna({
    "Postal Code":0,
    "Product Name":"Unknown Product"
})

In [15]:
df.show(10)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [16]:
#Rename columns by replacing spaces with underscores

for col_name in df.columns:
    df = df.withColumnRenamed(
        col_name,
        col_name.replace(" ", "_")
    )

df.printSchema()

root
 |-- Row_ID: integer (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: string (nullable = true)
 |-- Ship_Date: string (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: integer (nullable = false)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product_Name: string (nullable = false)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



In [17]:
delta_path = r"C:\Users\Soham Deshmukh\Desktop\B.Tech\Celebal Internship\cei-data-engineering\WEEK-7\delta\superstore"

(
    df.write
      .format("delta")
      .mode("overwrite")
      .save(delta_path)
)

In [18]:
spark.read.format("delta").load(delta_path).show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row_ID|      Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|  Customer_Name|  Segment|      Country|           City|     State|Postal_Code|Region|     Product_ID|       Category|Sub-Category|        Product_Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [19]:
#required due to the data type mismatch in the source and target delta table
df = (
    df.withColumn("Sales", col("Sales").cast(DoubleType()))
      .withColumn("Profit", col("Profit").cast(DoubleType()))
      .withColumn("Discount", col("Discount").cast(DoubleType()))
      .withColumn("Quantity", col("Quantity").cast(IntegerType()))
      .withColumn("Order_Date", to_date(col("Order_Date"), "M/d/yyyy"))
      .withColumn("Ship_Date", to_date(col("Ship_Date"), "M/d/yyyy"))
)

In [20]:
updates = (
    df.limit(5)
      .withColumn("Sales", col("Sales") + 100)
      .withColumn("Profit", col("Profit") + 20)
)
updates.show(truncate=False)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+-----------------------------------------------------------+------------------+--------+--------+--------+
|Row_ID|Order_ID      |Order_Date|Ship_Date |Ship_Mode     |Customer_ID|Customer_Name  |Segment  |Country      |City           |State     |Postal_Code|Region|Product_ID     |Category       |Sub-Category|Product_Name                                               |Sales             |Quantity|Discount|Profit  |
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+-----------------------------------------------------------+------------------+--------+--------+--------+
|1     |CA-2016-152156|2016-11-08|2016-11-11|Second Class  |CG-12520  

In [25]:
new_records = (
    df.limit(5)
      .withColumn("Row_ID", col("Row_ID") + 10000)
      .withColumn("Order_ID", concat(lit("NEW-"), col("Order_ID")))
)

In [26]:
incremental_df = updates.unionByName(new_records)
print("Incremental Records:", incremental_df.count())
incremental_df.show(truncate=False)

Incremental Records: 10
+------+------------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+-----------------------------------------------------------+------------------+--------+--------+--------+
|Row_ID|Order_ID          |Order_Date|Ship_Date |Ship_Mode     |Customer_ID|Customer_Name  |Segment  |Country      |City           |State     |Postal_Code|Region|Product_ID     |Category       |Sub-Category|Product_Name                                               |Sales             |Quantity|Discount|Profit  |
+------+------------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+-----------------------------------------------------------+------------------+--------+--------+--------+
|1     |CA-2016-152156    |2016-11

In [27]:
deltaTable = DeltaTable.forPath(
    spark,
    delta_path
)

In [ ]:
(
    deltaTable.alias("target")
    .merge( #here we are merging the incremental_df with the existing delta table
        incremental_df.alias("source"),
        "target.Row_ID = source.Row_ID"
    )
    .whenMatchedUpdate(
        set={ #here we are updating the existing records in the delta table with the new values from the incremental_df
            "Customer_Name": "source.Customer_Name",
            "Sales": "source.Sales",
            "Profit": "source.Profit",
            "Quantity": "source.Quantity",
            "Discount": "source.Discount"
        }
    )
    .whenNotMatchedInsertAll()
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [29]:
merge_result = (
    deltaTable.alias("target")
    .merge(
        incremental_df.alias("source"),
        "target.Row_ID = source.Row_ID"
    )
    .whenMatchedUpdate(
        set={
            "Customer_Name": "source.Customer_Name",
            "Sales": "source.Sales",
            "Profit": "source.Profit",
            "Quantity": "source.Quantity",
            "Discount": "source.Discount"
        }
    )
    .whenNotMatchedInsertAll()
    .execute()
)

merge_result.show(truncate=False)

+-----------------+----------------+----------------+-----------------+
|num_affected_rows|num_updated_rows|num_deleted_rows|num_inserted_rows|
+-----------------+----------------+----------------+-----------------+
|10               |10              |0               |0                |
+-----------------+----------------+----------------+-----------------+



In [30]:
final_df = spark.read.format("delta").load(delta_path)

In [31]:
final_df.show(20, truncate=False)

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+----------------------------------------------------------------------------+-------+--------+--------+----------+
|Row_ID|Order_ID      |Order_Date|Ship_Date |Ship_Mode     |Customer_ID|Customer_Name     |Segment    |Country      |City           |State         |Postal_Code|Region |Product_ID     |Category       |Sub-Category|Product_Name                                                                |Sales  |Quantity|Discount|Profit    |
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+----------------------------------------------------------------------------+-------+--------+--------+----------+
|1     |CA-2016-

In [32]:
final_df.printSchema()


root
 |-- Row_ID: integer (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: string (nullable = true)
 |-- Ship_Date: string (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



In [33]:
#Display the total number of records in the final delta table after the merge operation
print("Total Records:", final_df.count())

Total Records: 9999


In [34]:
#Display the total number of unique Row_IDs in the final delta table after the merge operation
print(
    "Unique Row_ID:",
    final_df.select("Row_ID").distinct().count()
)

Unique Row_ID: 9999


In [35]:
#Check for duplicate Row_IDs in the final delta table after the merge operation
(
    final_df.groupBy("Row_ID")
            .count()
            .filter("count > 1")
            .show()
)

+------+-----+
|Row_ID|count|
+------+-----+
+------+-----+



In [36]:
#Check for null values in the final delta table after the merge operation
final_df.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in final_df.columns
]).show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row_ID|Order_ID|Order_Date|Ship_Date|Ship_Mode|Customer_ID|Customer_Name|Segment|Country|City|State|Postal_Code|Region|Product_ID|Category|Sub-Category|Product_Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



In [37]:
#compare the number of records in the original dataset, incremental dataset, and final delta table after the merge operation
print("=" * 20)
print("MERGE Summary")
print("=" * 20)

print("Original Dataset Rows :", df.count())
print("Incremental Dataset Rows :", incremental_df.count())
print("Final Delta Table Rows :", final_df.count())

MERGE Summary
Original Dataset Rows : 9994
Incremental Dataset Rows : 10
Final Delta Table Rows : 9999


In [ ]:
#Display upadted records
final_df.filter(
    col("Row_ID").isin([1, 2, 3, 4, 5])
).show(truncate=False)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+-----------------------------------------------------------+------------------+--------+--------+--------+
|Row_ID|Order_ID      |Order_Date|Ship_Date |Ship_Mode     |Customer_ID|Customer_Name  |Segment  |Country      |City           |State     |Postal_Code|Region|Product_ID     |Category       |Sub-Category|Product_Name                                               |Sales             |Quantity|Discount|Profit  |
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+-----------------------------------------------------------+------------------+--------+--------+--------+
|2     |CA-2016-152156|11/8/2016 |11/11/2016|Second Class  |CG-12520  

In [39]:
#Display newly inserted records
final_df.filter(
    col("Order_ID").startswith("NEW-")
).show(truncate=False)

+------+------------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+-----------------------------------------------------------+--------+--------+--------+--------+
|Row_ID|Order_ID          |Order_Date|Ship_Date |Ship_Mode     |Customer_ID|Customer_Name  |Segment  |Country      |City           |State     |Postal_Code|Region|Product_ID     |Category       |Sub-Category|Product_Name                                               |Sales   |Quantity|Discount|Profit  |
+------+------------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+-----------------------------------------------------------+--------+--------+--------+--------+
|10003 |NEW-CA-2016-138688|2016-06-12|2016-06-16|Second Class  |DV-13045   |Darrin Van H

### Assignment Summary

In this assignment, I implemented an incremental data processing workflow using **PySpark** and **Delta Lake** with the Sample Superstore dataset. The following tasks were completed:

- Loaded the Sample Superstore dataset into a PySpark DataFrame.
- Explored the dataset by checking the schema, columns, and total number of records.
- Performed basic data cleaning by checking for missing values and duplicate records.
- Standardized the column names and converted important columns to the correct data types for Delta Lake.
- Saved the cleaned dataset as a Delta table.
- Created an incremental dataset to simulate new incoming data.
- Updated the values of existing records and added new records with unique identifiers.
- Applied the **Delta Lake MERGE** operation to update existing records and insert new records into the Delta table.
- Validated the final dataset by checking the row count, duplicate records, and null values.
- Displayed the final merged dataset and verified that the updates and inserts were applied successfully.
- Completed the assignment by demonstrating how Delta Lake supports efficient incremental data processing using the MERGE operation.